<a href="https://colab.research.google.com/github/Bavesh-08/CIFAR-10-using-vision-transformer/blob/main/CIFAR_10_using_vision_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Vision Transformer (ViT) - Built with TensorFlow / Keras
Dataset : CIFAR-10  (loaded directly from tf.keras.datasets)

Architecture overview
---------------------
Image → Patch Embedding → [CLS token + Positional Encoding]
      → N × Transformer Encoder Blocks
      → MLP Head → Class Probabilities
"""

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
# 1.  HYPER-PARAMETERS
# ─────────────────────────────────────────────────────────────
IMAGE_SIZE    = 32        # CIFAR-10 images are 32×32
PATCH_SIZE    = 4         # each patch is 4×4 pixels
NUM_PATCHES   = (IMAGE_SIZE // PATCH_SIZE) ** 2   # 64 patches
PROJECTION_DIM = 64       # dimension every patch is projected to
NUM_HEADS     = 4         # multi-head attention heads
TRANSFORMER_LAYERS = 4   # number of encoder blocks
MLP_DIM       = 128       # hidden size inside the feed-forward MLP
NUM_CLASSES   = 10        # CIFAR-10 has 10 classes
DROPOUT_RATE  = 0.1
BATCH_SIZE    = 64
EPOCHS        = 20
LEARNING_RATE = 1e-3

In [3]:
# 2.  DATA  (CIFAR-10 — no external download needed)
# ─────────────────────────────────────────────────────────────
def load_data():
    """Load and preprocess CIFAR-10."""
    (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

    # Normalize to [0, 1]
    x_train = x_train.astype("float32") / 255.0
    x_test  = x_test.astype("float32")  / 255.0

    # One-hot encode labels
    y_train = keras.utils.to_categorical(y_train, NUM_CLASSES)
    y_test  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

    print(f"Train : {x_train.shape}  |  Test : {x_test.shape}")
    return (x_train, y_train), (x_test, y_test)

In [4]:
# 3.  PATCH EMBEDDING LAYER
#     Splits each image into fixed-size patches and projects
#     every patch to PROJECTION_DIM dimensions.
# ─────────────────────────────────────────────────────────────
class PatchEmbedding(layers.Layer):
    """
    1. Extract non-overlapping patches from the image.
    2. Flatten each patch.
    3. Linearly project to PROJECTION_DIM.
    """
    def __init__(self, patch_size, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.patch_size     = patch_size
        self.projection_dim = projection_dim
        self.projection     = layers.Dense(projection_dim)

    def call(self, images):
        batch_size = tf.shape(images)[0]

        # Extract patches: (batch, num_patches, patch_h * patch_w * channels)
        patches = tf.image.extract_patches(
            images=images,
            sizes  =[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates  =[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]                     # flattened patch size
        patches    = tf.reshape(patches, [batch_size, -1, patch_dims])

        # Linear projection
        return self.projection(patches)                    # (batch, num_patches, proj_dim)

In [5]:
# 4.  POSITIONAL ENCODING
#     Adds a learnable position embedding + a learnable [CLS]
#     token to the patch sequence.
# ─────────────────────────────────────────────────────────────
class PositionalEncoding(layers.Layer):
    """
    Prepend a [CLS] token and add learnable positional embeddings.
    Output shape: (batch, num_patches + 1, projection_dim)
    """
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        # [CLS] token — one shared learnable vector
        self.cls_token = self.add_weight(
            "cls_token", shape=(1, 1, projection_dim), initializer="zeros"
        )
        # Positional embeddings — one per position (patches + CLS)
        self.pos_embedding = self.add_weight(
            "pos_embedding",
            shape=(1, num_patches + 1, projection_dim),
            initializer="random_normal",
        )

    def call(self, x):
        batch_size = tf.shape(x)[0]
        cls_tokens = tf.broadcast_to(self.cls_token, [batch_size, 1, x.shape[-1]])
        x = tf.concat([cls_tokens, x], axis=1)   # prepend CLS
        return x + self.pos_embedding             # add positional info


In [6]:
# 5.  TRANSFORMER ENCODER BLOCK
#     LayerNorm → Multi-Head Attention → residual
#     LayerNorm → MLP (Dense→GELU→Dense) → residual
# ─────────────────────────────────────────────────────────────
class TransformerBlock(layers.Layer):
    def __init__(self, projection_dim, num_heads, mlp_dim, dropout_rate, **kwargs):
        super().__init__(**kwargs)

        # --- Attention branch ---
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn  = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=projection_dim // num_heads,
            dropout=dropout_rate,
        )
        self.drop1 = layers.Dropout(dropout_rate)

        # --- Feed-Forward (MLP) branch ---
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp   = keras.Sequential([
            layers.Dense(mlp_dim, activation="gelu"),
            layers.Dropout(dropout_rate),
            layers.Dense(projection_dim),
            layers.Dropout(dropout_rate),
        ])

    def call(self, x, training=False):
        # Self-attention + residual
        x_norm = self.norm1(x)
        attn_out = self.attn(x_norm, x_norm, training=training)
        x = x + self.drop1(attn_out, training=training)

        # MLP + residual
        x_norm = self.norm2(x)
        x = x + self.mlp(x_norm, training=training)
        return x

In [7]:
# 6.  FULL VISION TRANSFORMER MODEL
# ─────────────────────────────────────────────────────────────
def build_vit(
    image_size    = IMAGE_SIZE,
    patch_size    = PATCH_SIZE,
    num_patches   = NUM_PATCHES,
    projection_dim= PROJECTION_DIM,
    num_heads     = NUM_HEADS,
    transformer_layers = TRANSFORMER_LAYERS,
    mlp_dim       = MLP_DIM,
    num_classes   = NUM_CLASSES,
    dropout_rate  = DROPOUT_RATE,
):
    inputs = keras.Input(shape=(image_size, image_size, 3))

    # Step 1 — Patch Embedding
    x = PatchEmbedding(patch_size, projection_dim, name="patch_embedding")(inputs)

    # Step 2 — Positional Encoding (adds CLS token)
    x = PositionalEncoding(num_patches, projection_dim, name="pos_encoding")(x)

    # Step 3 — Stack of Transformer Encoder Blocks
    for i in range(transformer_layers):
        x = TransformerBlock(
            projection_dim, num_heads, mlp_dim, dropout_rate,
            name=f"transformer_block_{i}"
        )(x)

    # Step 4 — Extract [CLS] token (index 0) and classify
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    cls_output = x[:, 0]                          # (batch, projection_dim)

    # MLP classification head
    x = layers.Dense(mlp_dim, activation="gelu")(cls_output)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="VisionTransformer")


In [8]:
# 7.  TRAINING
# ─────────────────────────────────────────────────────────────
def train():
    (x_train, y_train), (x_test, y_test) = load_data()

    model = build_vit()
    model.summary()

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    # Callbacks
    callbacks = [
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1),
    ]

    history = model.fit(
        x_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_split=0.1,
        callbacks=callbacks,
    )

    # Evaluate
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"\nTest accuracy : {test_acc:.4f}  |  Test loss : {test_loss:.4f}")

    return model, history


In [9]:
# 8.  QUICK INFERENCE DEMO
# ─────────────────────────────────────────────────────────────
CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

def predict_single(model, image: np.ndarray):
    """
    Predict the class of a single CIFAR-10 image.
    image : numpy array of shape (32, 32, 3), values in [0, 1]
    """
    img = np.expand_dims(image, axis=0)         # add batch dimension
    probs = model.predict(img, verbose=0)[0]
    pred_class = CLASS_NAMES[np.argmax(probs)]
    confidence = np.max(probs) * 100
    print(f"Prediction : {pred_class}  ({confidence:.1f}% confidence)")
    return pred_class, confidence

In [11]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

if __name__ == "__main__":
    # Redefine PositionalEncoding to fix the add_weight call
    # This redefinition must be here to comply with the instruction to modify ONLY JR-34FsLK2H3.
    # In a typical scenario, cell ZqqciIhpKjj9 would be directly modified.
    class PositionalEncoding(layers.Layer):
        """
        Prepend a [CLS] token and add learnable positional embeddings.
        Output shape: (batch, num_patches + 1, projection_dim)
        """
        def __init__(self, num_patches, projection_dim, **kwargs):
            super().__init__(**kwargs)
            # [CLS] token — one shared learnable vector
            self.cls_token = self.add_weight(
                name="cls_token", shape=(1, 1, projection_dim), initializer="zeros"
            )
            # Positional embeddings — one per position (patches + CLS)
            self.pos_embedding = self.add_weight(
                name="pos_embedding",
                shape=(1, num_patches + 1, projection_dim),
                initializer="random_normal",
            )

        def call(self, x):
            batch_size = tf.shape(x)[0]
            cls_tokens = tf.broadcast_to(self.cls_token, [batch_size, 1, x.shape[-1]])
            x = tf.concat([cls_tokens, x], axis=1)   # prepend CLS
            return x + self.pos_embedding             # add positional info

    model, history = train()

    # Demo prediction on one test image
    (_, _), (x_test, y_test) = keras.datasets.cifar10.load_data()
    x_test = x_test.astype("float32") / 255.0
    predict_single(model, x_test[0])


Train : (50000, 32, 32, 3)  |  Test : (10000, 32, 32, 3)


Model: "VisionTransformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ patch_embedding                 │ (None, None, 64)       │         3,136 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pos_encoding                    │ (None, 65, 64)         │         4,224 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_0             │ (None, 65, 64)         │        33,472 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 65, 64)         │        33,472 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, 65, 64)         │        33,472 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ (None, 65, 64)         │        33,472 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_8           │ (None, 65, 64)         │           128 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 150,986 (589.79 KB)

 Trainable params: 150,986 (589.79 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 55s 39ms/step - accuracy: 0.3053 - loss: 1.8308 - val_accuracy: 0.4148 - val_loss: 1.5825 - learning_rate: 0.0010
Epoch 2/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.4334 - loss: 1.5438 - val_accuracy: 0.4808 - val_loss: 1.4189 - learning_rate: 0.0010
Epoch 3/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.4783 - loss: 1.4265 - val_accuracy: 0.5194 - val_loss: 1.3185 - learning_rate: 0.0010
Epoch 4/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.5072 - loss: 1.3566 - val_accuracy: 0.5296 - val_loss: 1.2695 - learning_rate: 0.0010
Epoch 5/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.5273 - loss: 1.3013 - val_accuracy: 0.5528 - val_loss: 1.2422 - learning_rate: 0.0010
Epoch 6/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.5416 - loss: 1.2669 - val_accuracy: 0.5574 - val_loss: 1.2210 - learning_rate: 0.0010
Epoch 7/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.5540 - loss: 1